In [28]:
from census import Census
from pygris import block_groups
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union


In [29]:
# =====================================================
# CONFIG
# =====================================================

# --- ACS / block-group demographic data ---
API_KEY = open("../../data/census/api.txt").read().strip()

STATE_FIPS = "06"
COUNTY_FIPS = "001"
STATE_NAME = "CA"
COUNTY_NAME = "Alameda"

# Save the full county-wide block-group layer as an intermediate file too
# (handy for reuse / QA), independent of the BID catchment outputs below.
SAVE_INTERMEDIATE_BG = True
INTERMEDIATE_BG_PATH = "../../data/geo/block_groups/Alameda_BG.geojson"

# --- BID catchment area ---

# Path to the BID/corridor polygon(s). One row per BID. This file is
# read only to select and buffer around - it is NOT modified or
# re-saved; all aggregated metrics are attached to CATCHMENT_AREA
# instead (see below).
BID_PATH = "../../data/geo/block_groups/OAK_BIDs.geojson"

BUFFER_KM = 0.5
KM_TO_M = 1000
SQKM_TO_SQMI = 0.386102

# Equal-area, meters-based CRS used for buffering and all area math
PROJECTED_CRS = 3310

OUTPUT_CATCHMENT_AREA = "../../data/geo/block_groups/OAK_catchment_area.geojson"

# ---------------------------------------------------------------
# Variables to interpolate, split by how they should be aggregated:
#
# EXTENSIVE = counts/totals. Apportioned by the fraction of each
# block group's AREA that falls inside the 0.5km buffer (assumes the
# variable is spread uniformly across the block group), then summed
# across all intersecting block groups.
#
# INTENSIVE = rates/medians/averages. Aggregated as a population-
# weighted average across the intersecting block groups (weight =
# each block group's apportioned population), since summing a rate
# or a median across areas isn't meaningful.
#
# pop_density_sqmi and purchasing_power_density are recomputed from
# the aggregated extensive totals + the buffer's true area further
# down, rather than averaged - more accurate than interpolating a
# density directly.
# ---------------------------------------------------------------

EXTENSIVE_VARS = [
    "pop_count",
    "total_households",
    "total_housing_units",
    "purchasing_power_index",
]

INTENSIVE_VARS = [
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "per_capita_income",
    "avg_household_size",
    "median_age",
    "median_year_built",
    "drive_alone_share",
    "active_transit_share",
    "renter_share",
    "low_income_share",
]


In [30]:
# =====================================================
# ACS DATA
# =====================================================

c = Census(API_KEY, year=2024)

# Define your variables: code -> friendly column name
ACS_VARIABLES = {
    "B01003_001E": "pop_count",
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    "B11001_001E": "total_households",

    # Commuting / foot-traffic & parking-demand proxies (B08301)
    "B08301_001E": "commute_total",
    "B08301_003E": "drove_alone",
    "B08301_009E": "public_transit",
    "B08301_017E": "bicycle",
    "B08301_018E": "walked",

    # Vehicle access (B08201)
    "B08201_001E": "hh_vehicle_universe",
    "B08201_002E": "zero_vehicle_hh",

    # Tenure / displacement-risk exposure (B25003)
    "B25003_001E": "tenure_total",
    "B25003_003E": "renter_occupied",

    # Nativity / immigrant-entrepreneurship proxy (B05001)
    "B05001_001E": "citizenship_universe",
    "B05001_005E": "naturalized_citizen",
    "B05001_006E": "not_us_citizen",

    # Household income distribution, low-income brackets (B19001)
    "B19001_001E": "income_bracket_universe",
    "B19001_002E": "income_under_10k",
    "B19001_003E": "income_10_15k",
    "B19001_004E": "income_15_20k",
    "B19001_005E": "income_20_25k",
    "B19001_006E": "income_25_30k",
    "B19001_007E": "income_30_35k",

    # Industry of employed residents - retail/food-service tie to corridor (C24050)
    "C24050_001E": "employed_total",
    "C24050_006E": "retail_trade_employed",
    "C24050_012E": "arts_ent_accom_food_employed",

    # --- additional raw variables (context layers) ---
    "B25064_001E": "median_gross_rent",
    "B19301_001E": "per_capita_income",
    "B25010_001E": "avg_household_size",
    "B01002_001E": "median_age",
    "B25035_001E": "median_year_built",
    "B25002_001E": "total_housing_units",
}

print("Downloading ACS data...")

acs = c.acs5.state_county_blockgroup(
    tuple(ACS_VARIABLES.keys()),
    STATE_FIPS,
    COUNTY_FIPS,
    Census.ALL
)

df = pd.DataFrame(acs)

# Build GEOID
df["tract"] = df["tract"].str.zfill(6)
df["block group"] = df["block group"].str.zfill(1)
df["GEOID"] = (
    df["state"]
    + df["county"]
    + df["tract"]
    + df["block group"]
)

# Convert each variable to numeric and rename
for code, col_name in ACS_VARIABLES.items():
    df[col_name] = pd.to_numeric(df[code], errors="coerce")

# =====================================================
# HANDLE CENSUS "NO DATA" SENTINELS
# =====================================================
# The Census API encodes suppressed/unavailable estimates as -666666666.
# Treat that sentinel as missing across every ACS field, and additionally
# treat pop_count == 0 as missing (a data-quality flag rather than a true
# zero population for a residential block group). Any derived metric that
# depends on one of these null fields becomes null automatically once the
# inputs are set to NaN, since NaN propagates through arithmetic.
census_value_cols = list(ACS_VARIABLES.values())
df[census_value_cols] = df[census_value_cols].mask(df[census_value_cols] == -666666666)
df["pop_count"] = df["pop_count"].mask(df["pop_count"] == 0)

# Drop raw census code columns, keep only friendly names + GEOID
raw_cols = list(ACS_VARIABLES.keys()) + ["state", "county", "tract", "block group"]
df = df.drop(columns=raw_cols)


In [31]:
# =====================================================
# BLOCK GROUP GEOMETRIES + JOIN
# =====================================================

print("Downloading block group geometries...")

bg = block_groups(
    state=STATE_NAME,
    county=COUNTY_NAME,
    year=2024
)

acs_cols = ["GEOID"] + list(ACS_VARIABLES.values())

gdf = bg.merge(
    df[acs_cols],
    on="GEOID",
    how="left"
)

cols_to_drop = [
    "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE",
    "GEOIDFQ", "NAMELSAD", "MTFCC", "FUNCSTAT",
    "ALAND", "AWATER", "INTPTLON", "INTPTLAT"
]

gdf = gdf.drop(columns=cols_to_drop)

gdf.head()


Using FIPS code '06' for input 'CA'
Using FIPS code '001' for input 'Alameda'


,GEOID,geometry,pop_count,median_household_income,median_home_value,total_households,commute_total,drove_alone,public_transit,bicycle,...,income_30_35k,employed_total,retail_trade_employed,arts_ent_accom_food_employed,median_gross_rent,per_capita_income,avg_household_size,median_age,median_year_built,total_housing_units
0,060014423012,"POLYGON ((-121.96678 37.5303, -121.96678 37.53...",1173.0,102617.0,1326900.0,425.0,616.0,410.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2454.0,39663.0,2.76,41.2,1976.0,497.0
1,060014060001,"POLYGON ((-122.26838 37.78803, -122.26736 37.7...",2035.0,65425.0,784200.0,1058.0,1151.0,541.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2116.0,52351.0,1.88,39.4,2014.0,1131.0
2,060014337001,"POLYGON ((-122.11467 37.6887, -122.11462 37.68...",1508.0,91875.0,786400.0,419.0,817.0,519.0,0.0,0.0,...,29.0,NaN,NaN,NaN,1993.0,30208.0,3.58,33.4,1960.0,445.0
3,060014011004,"POLYGON ((-122.26764 37.82783, -122.2676 37.82...",2675.0,98570.0,781300.0,1215.0,1324.0,423.0,0.0,0.0,...,17.0,NaN,NaN,NaN,2602.0,69220.0,2.17,32.2,2013.0,1231.0
4,060014012001,"POLYGON ((-122.26016 37.83119, -122.26003 37.8...",1443.0,235109.0,1344400.0,686.0,927.0,222.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2802.0,113397.0,2.10,38.0,1958.0,686.0


In [32]:
# =====================================================
# DENSITY CALCULATION
# =====================================================

gdf = gdf.to_crs(PROJECTED_CRS)

gdf["area_sqkm"] = (
    gdf.geometry.area / 1_000_000
)

gdf["pop_density"] = (
    gdf["pop_count"] /
    gdf["area_sqkm"]
)

gdf = gdf.to_crs(4326)


In [33]:
# =====================================================
# VARIABLE 1: DER-POP-01 - Population density (per sq. mi.)
# =====================================================
# Foot-traffic / walkability context - area_sqkm computed in the DENSITY cell above.

gdf["area_sqmi"] = gdf["area_sqkm"] * SQKM_TO_SQMI
gdf["pop_density_sqmi"] = gdf["pop_count"] / gdf["area_sqmi"]


In [34]:
# =====================================================
# VARIABLE 2: DER-INC-05 - Aggregate purchasing power index
# =====================================================
# Market-sizing layer for retail siting / gap analysis.
# Formula: median household income x total households (by block group).

gdf["purchasing_power_index"] = gdf["median_household_income"] * gdf["total_households"]


In [35]:
# =====================================================
# VARIABLE 3: DER-INC-06 - Purchasing power density (per sq. mi.)
# =====================================================
# Highlights where corridor market demand is spatially densest.

gdf["purchasing_power_density"] = gdf["purchasing_power_index"] / gdf["area_sqmi"]


In [36]:
# =====================================================
# VARIABLE 4: DER-TRAN-01 - Drive-alone commute share
# =====================================================
# Baseline auto-dependency measure, useful for parking-demand planning near the corridor.
# Formula: B08301_003 (drove alone) / B08301_001 (total workers 16+).

gdf["drive_alone_share"] = gdf["drove_alone"] / gdf["commute_total"]


In [37]:
# =====================================================
# VARIABLE 5: DER-TRAN-02 - Active/transit commute share (walk + bike + transit)
# =====================================================
# Strong proxy for street-level foot traffic and storefront visibility.
# NOTE: verify B08301 sub-cell numbers (009/017/018) against the Census variables
# list for your ACS vintage - the table has ~20 categories and numbering can shift.

gdf["active_transit_share"] = (
    gdf["public_transit"] + gdf["bicycle"] + gdf["walked"]
) / gdf["commute_total"]


In [38]:
# =====================================================
# VARIABLE 7: DER-HSG-01 - Renter share
# =====================================================
# Renters are typically more exposed to displacement from corridor investment.
# Formula: B25003_003 (renter-occupied) / B25003_001 (total occupied housing units).

gdf["renter_share"] = gdf["renter_occupied"] / gdf["tenure_total"]


In [39]:
# =====================================================
# VARIABLE 9: DER-INC-01 - Low-income household share (<$35,000)
# =====================================================
# Identifies concentrations of economically vulnerable households - relevant to
# retail mix and affordability along the corridor.
# Formula: sum of B19001 brackets under $35,000 / B19001_001 (total households).

gdf["low_income_share"] = (
    gdf["income_under_10k"] + gdf["income_10_15k"] + gdf["income_15_20k"]
    + gdf["income_20_25k"] + gdf["income_25_30k"] + gdf["income_30_35k"]
) / gdf["income_bracket_universe"]


In [40]:
# =====================================================
# RAW VARIABLE: Median gross rent (B25064_001)
# =====================================================
# Commercial-adjacent affordability signal; pairs with the purchasing-power layers.

gdf["median_gross_rent"] = gdf["median_gross_rent"].round(0)


In [41]:
# =====================================================
# RAW VARIABLE: Per capita income (B19301_001)
# =====================================================
# Complements median household income where household size varies widely
# across the corridor.

gdf["per_capita_income"] = gdf["per_capita_income"].round(0)


In [42]:
# =====================================================
# RAW VARIABLE: Average household size (B25010_001)
# =====================================================
# Used to translate population counts into household-level demand estimates.

gdf["avg_household_size"] = gdf["avg_household_size"].round(2)


In [43]:
# =====================================================
# RAW VARIABLE: Median age (B01002_001)
# =====================================================
# Signals whether the corridor's customer/labour base skews younger
# (nightlife, retail) or older (healthcare, personal services).

gdf["median_age"] = gdf["median_age"].round(1)


In [44]:
# =====================================================
# RAW VARIABLE: Median year structure built (B25035_001)
# =====================================================
# Proxy for building-stock age, relevant to storefront condition and
# redevelopment potential along the corridor.

gdf["median_year_built"] = gdf["median_year_built"].round(0)


In [45]:
# =====================================================
# CLEAN UP INTERMEDIATE COLUMNS
# =====================================================
# Drop the raw numerator/denominator columns now that the derived rates/indexes
# above have been calculated from them, keeping the friendly final metrics.

intermediate_cols = [
    "commute_total", "drove_alone", "public_transit", "bicycle", "walked",
    "hh_vehicle_universe", "zero_vehicle_hh",
    "tenure_total", "renter_occupied",
    "citizenship_universe", "naturalized_citizen", "not_us_citizen",
    "income_bracket_universe", "income_under_10k", "income_10_15k", "income_15_20k",
    "income_20_25k", "income_25_30k", "income_30_35k",
    "retail_trade_employed", "arts_ent_accom_food_employed",
    "employed_total",
]

gdf = gdf.drop(columns=[c for c in intermediate_cols if c in gdf.columns])


In [46]:
# =====================================================
# SAVE INTERMEDIATE BLOCK GROUP LAYER (optional)
# =====================================================

if SAVE_INTERMEDIATE_BG:
    gdf.to_file(INTERMEDIATE_BG_PATH, driver="GeoJSON")


In [47]:
# =====================================================
# LOAD BID + BUILD 1KM BUFFERS
# =====================================================

bids = gpd.read_file(BID_PATH)

# Reproject everything to a common, equal-area, meters-based CRS for
# buffering and area calculations
bids_proj = bids.to_crs(PROJECTED_CRS)
bg_proj = gdf.to_crs(PROJECTED_CRS)

buffer_dist_m = BUFFER_KM * KM_TO_M
bids_proj["buffer_geom"] = bids_proj.geometry.buffer(buffer_dist_m)


In [48]:
# =====================================================
# AGGREGATE SELECTED BLOCK GROUPS
# =====================================================
# The 1km buffer is used ONLY to select block groups (any intersection
# at all counts as "in the catchment"). Once selected, each block
# group's FULL value is used - no partial-overlap/area-fraction
# weighting - since the catchment itself is defined as the whole
# extent of the selected block groups (see CATCHMENT_AREA below).
#
#   - Extensive variables (totals): simple sum across selected block groups.
#   - Intensive variables (rates/medians): population-weighted average
#     across selected block groups, weighted by each block group's
#     full pop_count.

def aggregate_catchment(buffer_geom, bg_proj, extensive_vars, intensive_vars):
    selected = bg_proj[bg_proj.intersects(buffer_geom)].copy()

    if selected.empty:
        result = {v: np.nan for v in extensive_vars + intensive_vars}
        result["n_block_groups"] = 0
        return result, selected

    result = {"n_block_groups": len(selected)}

    # Extensive: simple sum of full values
    for var in extensive_vars:
        result[var] = selected[var].sum(min_count=1)

    # Intensive: population-weighted average using each block group's
    # full pop_count as the weight
    weight = selected["pop_count"]

    for var in intensive_vars:
        valid = selected[var].notna() & weight.notna() & (weight > 0)
        if valid.sum() == 0:
            result[var] = np.nan
        else:
            result[var] = np.average(selected.loc[valid, var], weights=weight[valid])

    return result, selected


In [49]:
# =====================================================
# RUN AGGREGATION FOR EACH BID
# =====================================================

records = []
dissolved_geoms = []  # full extent of each BID's selected block groups

for _, row in bids_proj.iterrows():
    buffer_geom = row["buffer_geom"]
    metrics, selected = aggregate_catchment(buffer_geom, bg_proj, EXTENSIVE_VARS, INTENSIVE_VARS)
    records.append(metrics)

    if selected.empty:
        dissolved_geoms.append(buffer_geom)  # fallback: nothing selected
    else:
        dissolved_geoms.append(unary_union(selected.geometry.tolist()))

metrics_df = pd.DataFrame(records)

# Area of the actual catchment (the dissolved, whole selected block
# groups) - not the buffer - since that's what the metrics now represent
metrics_df["area_sqkm"] = [g.area / 1_000_000 for g in dissolved_geoms]
metrics_df["area_sqmi"] = metrics_df["area_sqkm"] * SQKM_TO_SQMI

metrics_df["pop_density_sqmi"] = metrics_df["pop_count"] / metrics_df["area_sqmi"]
metrics_df["purchasing_power_density"] = (
    metrics_df["purchasing_power_index"] / metrics_df["area_sqmi"]
)


In [50]:
# =====================================================
# BUILD CATCHMENT_AREA (with metrics attached, keeping BID properties)
# =====================================================
# The catchment for each BID is a single polygon: the dissolved union
# of every block group selected above (dissolved_geoms) - the whole
# block groups, not clipped to the buffer edge - so the output
# boundary follows real census geography instead of a smooth circle,
# and matches exactly what the metrics were aggregated over.
#
# The aggregated metrics live here (not on the input BID_PATH file),
# alongside every existing BID column, so BID_PATH itself stays
# untouched.

CATCHMENT_AREA = bids_proj.drop(columns=["geometry", "buffer_geom"]).copy()
CATCHMENT_AREA = pd.concat(
    [CATCHMENT_AREA.reset_index(drop=True), metrics_df.reset_index(drop=True)],
    axis=1,
)
CATCHMENT_AREA["geometry"] = dissolved_geoms
CATCHMENT_AREA = gpd.GeoDataFrame(CATCHMENT_AREA, geometry="geometry", crs=PROJECTED_CRS)
CATCHMENT_AREA = CATCHMENT_AREA.to_crs(bids.crs)

CATCHMENT_AREA.head()

,FID,BID,Shp__Ar,Shp__Ln,n_block_groups,pop_count,total_households,total_housing_units,purchasing_power_index,median_household_income,...,median_year_built,drive_alone_share,active_transit_share,renter_share,low_income_share,area_sqkm,area_sqmi,pop_density_sqmi,purchasing_power_density,geometry
0,1,Downtown,669747.800781,4944.705216,31,28897.0,16686.0,19149.0,1.394134e+09,91444.621732,...,1980.887082,0.277770,0.028250,0.830693,0.319104,6.957112,2.686155,10757.756360,5.190075e+08,"POLYGON ((-122.27226 37.79313, -122.27269 37.7..."
1,2,Chinatown,987399.226562,4847.190191,27,26988.0,15465.0,17483.0,1.300376e+09,95195.300822,...,1982.471432,0.312378,0.036302,0.836568,0.280858,7.147247,2.759566,9779.797657,4.712249e+08,"POLYGON ((-122.27119 37.78949, -122.27388 37.7..."
2,3,Lake Merritt,923483.507812,5909.055754,35,40931.0,23919.0,27562.0,2.117287e+09,97996.749461,...,1974.433925,0.314075,0.030953,0.857098,0.265453,6.100082,2.355254,17378.594550,8.989634e+08,"POLYGON ((-122.26378 37.79963, -122.26384 37.7..."
3,4,Koreatown/Northgate,573652.921875,7483.464481,33,39949.0,21241.0,24505.0,1.929302e+09,102781.467760,...,1971.429197,0.336513,0.045087,0.831799,0.261345,7.238210,2.794687,14294.622120,6.903461e+08,"POLYGON ((-122.27168 37.80368, -122.27177 37.8..."
4,5,Montclair,73144.023438,1706.295224,6,9217.0,3700.0,3795.0,8.885725e+08,239096.649995,...,1954.037322,0.365632,0.017482,0.104879,0.053783,7.196779,2.778691,3317.030002,3.197810e+08,"POLYGON ((-122.21418 37.81305, -122.21408 37.8..."


In [51]:
# =====================================================
# SAVE
# =====================================================
# BID_PATH is left exactly as provided - only CATCHMENT_AREA (with the
# aggregated metrics attached) is written out.

CATCHMENT_AREA.to_file(OUTPUT_CATCHMENT_AREA, driver="GeoJSON")